In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
sns.set_theme(style="whitegrid")
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

# Importando os arquivos e carregando com Pandas

caminho_departamentos = 'dados/departamentos.csv'
dfDepartamentos = pd.read_csv(caminho_departamentos)

caminho_funcionarios = 'dados/funcionarios.csv'
dfFuncionarios = pd.read_csv(caminho_funcionarios)

caminho_filiais = 'dados/filiais.csv'
dfFiliais = pd.read_csv(caminho_filiais)


# 2.Análise Exploratória de Dados e Achados

## 2.1. Validação de Escopo e Limpeza dos Dados

In [ ]:
# Exploração Inicial dos Dados

print("--- Primeiras linhas de DEPARTAMENTOS ---")
display(dfDepartamentos.head())

print("\n--- Primeiras linhas de FUNCIONÁRIOS ---")
display(dfFuncionarios.head())

print("\n--- Primeiras linhas de FILIAIS ---")
display(dfFiliais.head())

## Analisando os diferentes tipos de respostas por coluna:

print("Respostas diferentes por coluna em funcionarios.csv")

for col in dfFuncionarios:
    print(dfFuncionarios[col].unique())



Problemas notados nos arquivos:

1. O salário_base é formatado com R$ e vírgula;

2. Formato das datas nas colunas 'data_admissao' e 'data_promocao' não está padronizado;

3. data_promocao retorna NaN (já que a célula está vazia) se o funcionário não foi promovido;

4. Certos funcionários não deram score de satisfação;

5. Os valores das células na coluna 'id_reporta_a' estão vazios para os sócios;

6. Gênero 'fem' ao invés de feminino;

---
Soluções propostas

1. Tirar o R$ e trocar a vírgula por ponto;

2. Padronizar as datas no formato YYYY-MM-DD (Ano, mês, dia);

3. Criar coluna 'foi_promovido' (0 se não foi promovido, 1 se foi)

4. Adicionar uma coluna ao que informa se o funcionário deu
score de satisfação ou não (0 se não deu score, 1 se deu) e substituir a célula vazia pela mediana do grupo que funcionário faz parte
(Junior, Pleno ou Senior | Ativo ou Desligado).;

5. Alterar os valores de 'id_reporta_a' das células dos sócios para 0 (para facilitar operações matemáticas no futuro);

6. Trocar 'fem' por 'feminino' em gênero;

---

 Outras inconsistências notadas:

  1. Muitos funcionários reportam à Ana Silva (ID 501), porém o funcionário é de nível júnior e não um sócio ou sênior.

  2. Nas planilhas não se informa a data de demissão. Seria um dado muito importante de se ter em mãos.

In [ ]:
# Tratamento de Dados

## 1. Tirar o R$ e trocar a vírgula por ponto para converter a coluna salario_base para número

dfFuncionarios['salario_base'] = dfFuncionarios['salario_base'].str.replace('R$', '')
dfFuncionarios['salario_base'] = dfFuncionarios['salario_base'].str.replace(',', '.')
dfFuncionarios['salario_base'] = pd.to_numeric(dfFuncionarios['salario_base'])

## 2. Padronizando as datas

dfFuncionarios['data_admissao'] = pd.to_datetime(
    dfFuncionarios['data_admissao'],
    dayfirst=True,
    format='mixed',
    errors='coerce'
)

''' Nas células dos funcionários que não foram promovidos, o valor da será NaT (Not a Time) '''
dfFuncionarios['data_promocao'] = pd.to_datetime(
    dfFuncionarios['data_promocao'],
    dayfirst=True,
    format='mixed',
    errors='coerce'
)

## 3. Implementar flag ('foi_promovido')

dfFuncionarios['foi_promovido'] = dfFuncionarios['data_promocao'].notna().astype(int)

## 4. Criar flag 'respondeu_pesquisa' e substituir a célula vazia pela mediana do grupo que o funcionário faz parte.

dfFuncionarios['respondeu_pesquisa'] = dfFuncionarios['score_satisfacao'].notna().astype(int)
dfFuncionarios['score_satisfacao'] = dfFuncionarios.groupby(['status_atual', 'nivel'])['score_satisfacao'].transform(lambda x: x.fillna(x.median()))

## 5. Alterar os valores de 'id_reporta_a' das células dos sócios para 0

dfFuncionarios.loc[dfFuncionarios['nivel'] == 'Sócio', 'id_reporta_a'] = 0

## 6. Trocar 'Fem' por 'Feminino'

dfFuncionarios['genero'] = dfFuncionarios['genero'].replace(['Fem', 'Femininoinino'], 'Feminino')

## Cruzamento de Dados e Engenharia de Atributos

### Merge indicando os departamentos dos funcionários

df_completo = dfFuncionarios.merge(
    dfDepartamentos[['id_departamento', 'nome_departamento']],
    on='id_departamento',
    how='left'
)

### Cruzando com filiais

df_completo = df_completo.merge(
    dfFiliais[['id_filial', 'cidade', 'estado']],
    on='id_filial',
    how='left'
)

''' display(df_completo.loc[df_completo['id_colaborador'] == 647].T) '''


display(df_completo)



## 2.2. Panorama Geral e Perfil Demográfico






In [ ]:
## Dados gerais do escritório

''' Visão geral da empresa antes de se iniciar o trabalho investigativo '''

### Total de colaboradores

total_colaboradores = len(df_completo)
print(f"Total de colaboradores analisados: {total_colaboradores}")

### Número de pessoas por sexo

contagem_genero = df_completo["genero"].value_counts()

rotulos_legenda_generos = [f"{genero}: {qtd} pessoas" for genero, qtd in contagem_genero.items()]

plt.figure(figsize=(6, 6))

fatias, textos, porcentagens = plt.pie(
    contagem_genero,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#4C72B0", "#DD8452", "#55A868"],
    textprops={"color": "white", "weight": "bold"}
)

plt.legend(
    fatias,
    rotulos_legenda_generos,
    title="Valores Absolutos",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    title_fontsize=12
)

plt.title("Distribuição de Colaboradores por Gênero", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("2.2.grafico_genero.png", dpi=300, bbox_inches="tight")
plt.show()

### Número de júniores, plenos, sêniores e sócios

contagem_nivel = df_completo["nivel"].value_counts()

rotulos_legenda_nivel = [f"{nivel}: {qtd} pessoas" for nivel, qtd in contagem_nivel.items()]

plt.figure(figsize=(8, 6))

fatias, textos, porcentagens = plt.pie(
    contagem_nivel,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#10b981", "#f59e0b", "#EF4444", "#4f46e5"],
    textprops={"color": "white", "weight": "bold"}
)

plt.legend(
    fatias,
    rotulos_legenda_nivel,
    title="Valores Absolutos",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    title_fontsize=12
)

plt.title("Distribuição de Colaboradores por Nível", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("2.2.grafico_nivel.png", dpi=300, bbox_inches="tight")
plt.show()

### Pessoas por departamento

contagem_departamento = df_completo["nome_departamento"].value_counts()

rotulos_legenda_departamento = [f"{departamento}: {qtd} pessoas" for departamento, qtd in contagem_departamento.items()]

fatias, textos, porcentagens = plt.pie(
    contagem_departamento,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#10b981", "#f59e0b", "#EF4444", "#4f46e5"],
    textprops={"color": "white", "weight": "bold"}
)

plt.legend(
    fatias,
    rotulos_legenda_departamento,
    title="Valores Absolutos",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    title_fontsize=12
)

plt.title("Distribuição de Colaboradores por Departamento", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("2.2.grafico_departamentos.png", dpi=300, bbox_inches="tight")
plt.show()

### Pessoas por Filial (Cidade)
contagem_filial = df_completo["cidade"].value_counts()

rotulos_legenda_filial = [f"{cidade}: {qtd} pessoas" for cidade, qtd in contagem_filial.items()]

fatias, textos, porcentagens = plt.pie(
    contagem_filial,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#10b981", "#f59e0b", "#EF4444", "#4f46e5"],
    textprops={"color": "white", "weight": "bold"}
)

plt.legend(
    fatias,
    rotulos_legenda_filial,
    title="Valores Absolutos",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    title_fontsize=1
)

plt.title("Distribuição de Colaboradores por Filial", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("2.2.grafico_filial.png", dpi=300, bbox_inches="tight")
plt.show()


### Número de pessoas promovidas

quantidade_promovidos = (df_completo["foi_promovido"] == 1).sum()
print(f"Quantidade de funcionários promovidos: {quantidade_promovidos}")

### Distribuição de pessoas ativas e desligadas

contagem_status = df_completo["status_atual"].value_counts()

rotulos_legenda_status = [f"{status}: {qtd} pessoas" for status, qtd in contagem_status.items()]

fatias, textos, porcentagens = plt.pie(
    contagem_status,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#4C72B0", "#EF4444"],
    textprops={"color": "white", "weight": "bold"}
)

plt.legend(
    fatias,
    rotulos_legenda_status,
    title="Valores Absolutos",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=11,
    title_fontsize=1
)

plt.title("Distribuição de Colaboradores por Status", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("2.2.grafico_status.png", dpi=300, bbox_inches="tight")
plt.show()

### Taxa geral de evasão (turnover) (%)

quantidade_desligados = (df_completo["status_atual"] == "Desligado").sum()

quantidade_ativos = (df_completo["status_atual"] == "Ativo").sum()

taxa_turnover = (quantidade_desligados / total_colaboradores) * 100

print(f"Taxa geral de evasão: {taxa_turnover:.2f}%")

### Distribuição de Home Office e Presencial

total_home_office = (df_completo["home_office"] == True).sum()
total_presencial = (df_completo["home_office"] == False).sum()

print(f"Funcionários em Home Office: {total_home_office}")
print(f"Funcionários Presenciais: {total_presencial}")

## Extraindo os dados em uma tabela

def extrair_metricas(df, coluna):
    contagem = df[coluna].value_counts()
    percentual = df[coluna].value_counts(normalize=True) * 100
    return contagem, percentual

total_colaboradores = len(df_completo)

linhas_tabela = []

mapeamento = [
    ("Gênero", "genero"),
    ("Senioridade", "nivel"),
    ("Status", "status_atual"),
    ("Departamento", "nome_departamento"),
    ("Filial", "cidade"),
]

for nome_cat, col in mapeamento:
    counts, percents = extrair_metricas(df_completo, col)
    primeira = True
    for subgrupo in counts.index:
        cat_label = nome_cat if primeira else ""
        qtd = counts[subgrupo]
        pct = f"{percents[subgrupo]:.1f}%"
        linhas_tabela.append([cat_label, str(subgrupo), f"{qtd}", pct])
        primeira = False

ho_counts = df_completo["home_office"].value_counts()
ho_pcts = df_completo["home_office"].value_counts(normalize=True) * 100
linhas_tabela.append(
    ["Modalidade", "Home Office", f"{ho_counts.get(True, 0)}", f"{ho_pcts.get(True, 0):.1f}%"]
)
linhas_tabela.append(
    ["", "Presencial", f"{ho_counts.get(False, 0)}", f"{ho_pcts.get(False, 0):.1f}%"]
)

prom_counts = df_completo["foi_promovido"].value_counts()
prom_pcts = df_completo["foi_promovido"].value_counts(normalize=True) * 100
linhas_tabela.append(
    ["Carreira", "Promovido", f"{prom_counts.get(1, 0)}", f"{prom_pcts.get(1, 0):.1f}%"]
)
linhas_tabela.append(
    ["", "Não Promovido", f"{prom_counts.get(0, 0)}", f"{prom_pcts.get(0, 0):.1f}%"]
)

fig, ax = plt.subplots(figsize=(8, 10))
ax.axis("off")

col_labels = ["Categoria", "Subgrupo / Métrica", "Quantidade (N)", "Proporção (%)"]

tabela = ax.table(
    cellText=linhas_tabela,
    colLabels=col_labels,
    loc="center",
    cellLoc="center",
)

tabela.auto_set_font_size(False)
tabela.set_fontsize(10)
tabela.scale(1.2, 1.35)

for (row, col), cell in tabela.get_celld().items():
    if row == 0:
        cell.set_facecolor("#4C72B0")
        cell.set_text_props(weight="bold", color="white")
    else:
        cell.set_facecolor("#F2F4F8" if row % 2 == 0 else "white")
        if col in [0, 1]:
            cell.set_text_props(ha="left")

plt.title(
    f"Resumo do Panorama Geral da Empresa (Total: {total_colaboradores} Colaboradores)",
    fontsize=12,
    fontweight="bold",
    pad=15,
)

plt.tight_layout()

plt.savefig("2.2.tabela_resumo_panorama.png", dpi=300, bbox_inches="tight")
plt.show()


## 2.3. Identificação de Padrões e Correlações de Turnover



In [ ]:
## Explorando as possíveis causas que fazem funcionários desligarem (investigação)

''' Gerando dataframes de funcionários ativos e desligados '''

df_ativos = df_completo.query("status_atual == 'Ativo'").copy()
df_desligados = df_completo.query("status_atual == 'Desligado'").copy()

### 1. Desligados por gênero

fig, ax = plt.subplots(figsize=(7, 4))

sns.countplot(
    data=df_desligados,
    x="genero",
    hue= "genero",
    legend= False,
    order=df_desligados["genero"].value_counts().index,
    palette="Blues_r",
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    ax.bar_label(container, padding=3, fontweight="bold")

plt.ylim(0, df_desligados["genero"].value_counts().max() * 1.15)

plt.title("Total de Desligamentos por Gênero", fontsize=12, fontweight="bold")
plt.xlabel("Gênero")
plt.ylabel("Quantidade de Desligados")
plt.tight_layout()

plt.savefig("2.3.grafico_desligados_por_genero.png", dpi=300, bbox_inches="tight")
plt.show()

### 2. Ativos x desligados por nível

df_nivel = df_completo.copy()

df_nivel["nivel"] = pd.Categorical(
    df_nivel["nivel"],
    categories=["Júnior", "Pleno", "Sênior", "Sócio"],
    ordered=True,
)

fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=df_nivel,
    x="nivel",
    hue="status_atual",
    multiple="stack",
    hue_order=["Desligado", "Ativo"],
    palette=["#EF4444","#4C72B0"],
    shrink=0.7,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    valores = container.datavalues
    novos_rotulos = ["" if v == 0 else f"{int(v)}" for v in valores]

    ax.bar_label(
        container,
        labels=novos_rotulos,
        label_type="center",
        fontweight="bold",
        color="white",
        fontsize=10,
    )

plt.title("Distribuição de Colaboradores por Nível (Ativos vs. Desligados)", fontsize=13, fontweight="bold")
plt.xlabel("Nível de Senioridade", fontsize=11)
plt.ylabel("Quantidade de Funcionários", fontsize=11)

sns.move_legend(ax, "upper right", title="Status", frameon=True)
plt.tight_layout()

plt.savefig("2.3.grafico_nivel_status.png", dpi=300, bbox_inches="tight")
plt.show()

### 3. Mediana dos salários dos ativos x desligados por senioridade

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=df_completo,
    x="nivel",
    y="salario_base",
    hue="status_atual",
    order=["Júnior", "Pleno", "Sênior", "Sócio"],
    hue_order=["Ativo", "Desligado"],
    palette=["#4C72B0", "#EF4444"],
    estimator="mean",
    errorbar=None,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="R$ %.0f",
        padding=3,
        fontsize=9,
        fontweight="bold",
    )

maior_salario = df_completo.groupby(["nivel", "status_atual"])["salario_base"].mean().max()
plt.ylim(0, maior_salario * 1.15)

plt.title("Comparativo de Média Salarial: Ativos vs. Desligados", fontsize=13, fontweight="bold")
plt.xlabel("Nível de Senioridade", fontsize=11)
plt.ylabel("Média Salarial (R$)", fontsize=11)

plt.legend(title="Status", frameon=True)
plt.tight_layout()

plt.savefig("2.3.grafico_salario_comparativo.png", dpi=300, bbox_inches="tight")
plt.show()

### 4. Efeito de promoção

df_promo = df_completo.copy()
df_promo["status_promocao"] = df_promo["foi_promovido"].replace(
    {0: "Não Promovido", 1: "Promovido", False: "Não Promovido", True: "Promovido"}
)

fig, ax = plt.subplots(figsize=(7, 5))

sns.histplot(
    data=df_promo,
    x="status_promocao",
    hue="status_atual",
    multiple="stack",
    hue_order=["Desligado","Ativo"],
    palette=["#EF4444","#4C72B0"],
    shrink=0.5,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    valores = container.datavalues
    rotulos = [f"{int(v)}" if v > 0 else "" for v in valores]

    ax.bar_label(
        container,
        labels=rotulos,
        label_type="center",
        fontweight="bold",
        color="white",
        fontsize=10,
    )

plt.title("Volume de Colaboradores por Promoção (Ativos vs. Desligados)", fontsize=12, fontweight="bold")
plt.xlabel("Status de Promoção", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)

sns.move_legend(ax, "upper right", title="Status", frameon=True)
plt.tight_layout()

plt.savefig("2.3.grafico_efeito_promocao.png", dpi=300, bbox_inches="tight")
plt.show()

### 5. Efeito Home Office

df_ho = df_completo.copy()

df_ho["modelo_trabalho"] = np.where(
    df_ho["home_office"].astype(str).str.contains("1|True"),
    "Home Office",
    "Presencial",
)

fig, ax = plt.subplots(figsize=(7, 5))

sns.histplot(
    data=df_ho,
    x="modelo_trabalho",
    hue="status_atual",
    multiple="stack",
    hue_order=["Desligado", "Ativo"],
    palette={"Ativo": "#4C72B0", "Desligado": "#EF4444"},
    shrink=0.5,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    valores = container.datavalues
    rotulos = [f"{int(v)}" if v > 0 else "" for v in valores]
    ax.bar_label(
        container,
        labels=rotulos,
        label_type="center",
        fontweight="bold",
        color="white",
        fontsize=10,
    )

max_total = df_ho["modelo_trabalho"].value_counts().max()
plt.ylim(0, max_total * 1.12)

plt.title(
    "Volume de Colaboradores por Modelo de Trabalho (Ativos vs. Desligados)",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Modelo de Trabalho", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)

sns.move_legend(ax, "upper right", title="Status", frameon=True)
plt.tight_layout()

plt.savefig(
    "2.3.grafico_efeito_home_office.png", dpi=300, bbox_inches="tight"
)
plt.show()

### 6. Efeito dos processos

evasao_processos = pd.crosstab(df_completo['processos_actifs'], df_completo['status_atual'], normalize='index') * 100
print('--- Efeito dos processos ---')
display(evasao_processos.round(2))

df_processos = df_completo.copy()
df_processos["taxa_turnover"] = (df_processos["status_atual"] == "Desligado") * 100

fig, ax = plt.subplots(figsize=(10, 5))

sns.lineplot(
    data=df_processos,
    x="processos_actifs",
    y="taxa_turnover",
    estimator="mean",
    errorbar=None,
    color="#EF4444",
    marker="o",
    linewidth=2,
    markersize=6,
    ax=ax,
)

max_x = df_processos["processos_actifs"].max()
ax.set_xticks(np.arange(10, max_x + 5, 5))
ax.set_xlim(8, max_x + 2)

ax.set_yticks(np.arange(0, 90, 10))
ax.set_ylim(0, 85)

plt.title("Taxa de Turnover (%) por Qtd. de Processos", fontsize=13, fontweight="bold")
plt.xlabel("Quantidade de Processos", fontsize=11)
plt.ylabel("Taxa de Desligamento (%)", fontsize=11)

plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("2.3.grafico_efeito_processos.png", dpi=300, bbox_inches="tight")
plt.show()

### 7. Evasão de gestores (Gestor ruim faz funcionários desistirem?)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df_gestor = df_completo[df_completo["id_reporta_a"] != 0].copy()
df_gestor["id_gestor_str"] = (
    df_gestor["id_reporta_a"].astype(int).astype(str)
)

fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=df_gestor,
    x="id_gestor_str",
    hue="status_atual",
    multiple="stack",
    hue_order=["Desligado", "Ativo"],
    palette={"Ativo": "#4C72B0", "Desligado": "#EF4444"},
    shrink=0.5,
    alpha=1,
    ax=ax,
)

for container in ax.containers:
    valores = container.datavalues
    rotulos = [f"{int(v)}" if v > 0 else "" for v in valores]
    ax.bar_label(
        container,
        labels=rotulos,
        label_type="center",
        fontweight="bold",
        color="white",
        fontsize=10,
    )

maior_total = df_gestor["id_gestor_str"].value_counts().max()
plt.ylim(0, maior_total * 1.15)

plt.title(
    "Volume de Colaboradores por Gestor (Ativos vs. Desligados)",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("ID do Gestor", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)

sns.move_legend(ax, "upper right", title="Status", frameon=True)

plt.tight_layout()
plt.savefig("2.3.grafico_efeito_gestor.png", dpi=300, bbox_inches="tight")
plt.show()

### 8. Taxa de evasão por departamento (%)

df_dept = df_completo.copy()

fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=df_dept,
    x="nome_departamento",
    hue="status_atual",
    multiple="stack",
    hue_order=["Desligado", "Ativo"],
    palette={
        "Ativo": "#4C72B0",
        "Desligado": "#EF4444",
    },
    shrink=0.5,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    valores = container.datavalues
    rotulos = [f"{int(v)}" if v > 0 else "" for v in valores]
    ax.bar_label(
        container,
        labels=rotulos,
        label_type="center",
        fontweight="bold",
        color="white",
        fontsize=10,
    )

totais_por_dept = df_dept["nome_departamento"].value_counts()
taxas_turnover = df_dept.groupby("nome_departamento")["status_atual"].apply(
    lambda x: (x == "Desligado").mean() * 100
)

ordem_eixo_x = [tick.get_text() for tick in ax.get_xticklabels()]

for i, dept in enumerate(ordem_eixo_x):
    if dept in taxas_turnover:
        total = totais_por_dept[dept]
        taxa = taxas_turnover[dept]
        ax.text(
            x=i,
            y=total + (totais_por_dept.max() * 0.02),
            s=f"{taxa:.1f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
            color="#EF4444",
            fontsize=11,
        )

maior_total = totais_por_dept.max()
plt.ylim(0, maior_total * 1.15)

plt.title(
    "Volume de Funcionários e Taxa de Turnover (%) por Departamento",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Departamento", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)

sns.move_legend(
    ax,
    "upper left",
    bbox_to_anchor=(1.02, 1),
    title="Status",
    frameon=True,
)
plt.tight_layout()

plt.savefig("2.3.grafico_evasao_departamento.png", dpi=300, bbox_inches="tight")
plt.show()

### 9. Taxa de evasão por filial (%)

df_filial = df_completo.copy()

fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=df_filial,
    x="cidade",
    hue="status_atual",
    multiple="stack",
    hue_order=["Desligado", "Ativo"],
    palette={"Ativo": "#4C72B0", "Desligado": "#EF4444"},
    shrink=0.5,
    alpha=1,
    ax=ax,
)

for container in ax.containers:
    valores = container.datavalues
    rotulos = [f"{int(v)}" if v > 0 else "" for v in valores]
    ax.bar_label(
        container,
        labels=rotulos,
        label_type="center",
        fontweight="bold",
        color="white",
        fontsize=10,
    )

totais_por_filial = df_filial["cidade"].value_counts()
taxas_turnover = df_filial.groupby("cidade")["status_atual"].apply(
    lambda x: (x == "Desligado").mean() * 100
)

ordem_eixo_x = [tick.get_text() for tick in ax.get_xticklabels()]

for i, cidade in enumerate(ordem_eixo_x):
    if cidade in taxas_turnover:
        total = totais_por_filial[cidade]
        taxa = taxas_turnover[cidade]
        ax.text(
            x=i,
            y=total + (totais_por_filial.max() * 0.02),
            s=f"{taxa:.1f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
            color="#EF4444",
            fontsize=11,
        )

maior_total = totais_por_filial.max()
plt.ylim(0, maior_total * 1.15)

plt.title(
    "Volume de Funcionários e Taxa de Turnover (%) por Filial",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Filial", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)

sns.move_legend(
    ax,
    "upper left",
    bbox_to_anchor=(1.02, 1),
    title="Status",
    frameon=True,
)

plt.tight_layout()
plt.savefig("2.3.grafico_evasao_filial.png", dpi=300, bbox_inches="tight")
plt.show()

### 10. Mediana dos scores de satisfação dos ativos x desligados por senioridade

ordem_cargos = ["Júnior", "Pleno", "Sênior", "Sócio"]

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=df_completo,
    x="nivel",
    y="score_satisfacao",
    hue="status_atual",
    order=ordem_cargos,
    hue_order=["Ativo", "Desligado"],
    palette={"Ativo": "#4C72B0", "Desligado": "#EF4444"},
    estimator="mean",
    errorbar=None,
    alpha=1,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=3,
        fontsize=9,
        fontweight="bold",
    )

max_score = (
    df_completo.groupby(["nivel", "status_atual"])["score_satisfacao"]
    .mean()
    .max()
)
ax.set_ylim(0, max_score * 1.15)

ax.set_title(
    "Comparativo Score de Satisfação: Ativos vs. Desligados",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("Nível de Senioridade", fontsize=11)
ax.set_ylabel("Média dos Scores", fontsize=11)

sns.move_legend(
    ax,
    "upper left",
    bbox_to_anchor=(1.02, 1),
    title="Status",
    frameon=True,
)

plt.tight_layout()
plt.savefig("2.3.grafico_comparacao_scores.png", dpi=300, bbox_inches="tight")
plt.show()

### 11. Média de horas extra por mês dos ativos x desligados por senioridade

ordem_cargos = ["Júnior", "Pleno", "Sênior", "Sócio"]
ordem_status = ["Ativo", "Desligado"]

fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=df_completo,
    x="nivel",
    y="horas_extras_mes",
    hue="status_atual",
    order=ordem_cargos,
    hue_order=ordem_status,
    palette=["#4C72B0", "#EF4444"],
    estimator="mean",
    errorbar=None,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    ax.bar_label(
        container, fmt="%1.2f", padding=3, fontsize=9, fontweight="bold"
    )

media_max = (
    df_completo.groupby(["nivel", "status_atual"])["horas_extras_mes"]
    .mean()
    .max()
)
plt.ylim(0, media_max * 1.15)

plt.title(
    "Comparativo de Horas Extras por Mês: Ativos vs. Desligados",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel("Nível de Senioridade", fontsize=11)
plt.ylabel("Média de Horas Extras (h)", fontsize=11)
plt.legend(title="Status", frameon=True)

plt.tight_layout()
plt.savefig("2.3.grafico_comparacao_horas.png", dpi=300, bbox_inches="tight")
plt.show()













## 2.4. Investigação Diagnóstica e Teste de Hipóteses

In [ ]:
### Hipotése 1: O departamento trabalhista exige mais de seus funcionários, e por isso a taxa de evasão é maior.

#### Gráfico de processos por trabalhadores por departamento

df_analise = df_completo.copy()

df_analise["faixa_processos"] = df_analise["processos_actifs"].apply(
    lambda x: "> 45 Processos" if x > 45 else "<= 45 Processos"
)

fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=df_analise,
    x="nome_departamento",
    hue="faixa_processos",
    multiple="stack",
    hue_order=["> 45 Processos","<= 45 Processos"],
    palette=["#EF4444","#4C72B0"],
    shrink=0.55,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    valores = container.datavalues
    rotulos = [f"{int(v)}" if v > 0 else "" for v in valores]
    ax.bar_label(
        container,
        labels=rotulos,
        label_type="center",
        fontweight="bold",
        color="white",
        fontsize=10,
    )

plt.title(
    "Volume de Colaboradores por Carga de Processos e Departamento",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Departamento", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)

sns.move_legend(
    ax, "upper left", bbox_to_anchor=(1.02, 1), title="Carga de Processos"
)

plt.tight_layout()
plt.savefig("2.4.grafico_processos_departamento.png", dpi=300, bbox_inches="tight")
plt.show()

#### Gráfico de horas extras mensais por trabalhador por departamento

fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=df_completo,
    x="nome_departamento",
    y="horas_extras_mes",
    hue="nome_departamento",
    estimator="mean",
    errorbar=None,
    palette="Blues_d",
    legend=False,
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f h",
        padding=4,
        fontweight="bold",
        fontsize=10,
        color="#333333",
    )

media_max = df_completo.groupby("nome_departamento")["horas_extras_mes"].mean().max()
plt.ylim(0, media_max * 1.18)

plt.title(
    "Média Mensal de Horas Extras por Departamento",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Departamento", fontsize=10)
plt.ylabel("Média de Horas Extras (horas)", fontsize=10)

plt.tight_layout()
plt.savefig("2.4.grafico_media_horas_extras_departamento.png", dpi=300, bbox_inches="tight")
plt.show()


### Hipótese 2: Os juniores saem devido a sobrecarga de processos e horas extras.

#### Gráfico de tendência (lineplot) de score de satisfacao dos juniores x processos.

fig, ax = plt.subplots(figsize=(8, 5))

df_junior = df_completo[df_completo["nivel"] == "Júnior"]

sns.lineplot(
    data=df_junior,
    x="processos_actifs",
    y="score_satisfacao",
    hue="status_atual",
    palette=["#4C72B0", "#EF4444"],
    errorbar=None,
    ax=ax,
)

max_score = df_junior["score_satisfacao"].max()
plt.ylim(0, max_score * 1.15)

plt.title(
    "Juniores: Impacto da Carga de Processos no Score de Satisfação",
    fontsize=12,
    fontweight="bold",
)

plt.xlabel("Quantidade de Processos", fontsize=10)
plt.ylabel("Score de Satisfação", fontsize=10)
plt.legend(title="Status", frameon=True)

plt.tight_layout()
plt.savefig("2.4.grafico_satisfacao_processos_juniores.png", dpi=300, bbox_inches="tight")
plt.show()

#### Gráfico de tendência (lineplot) de score de satisfacao dos juniores x quantidade de horas extras mensais.

fig, ax = plt.subplots(figsize=(8, 5))

sns.lineplot(
    data=df_junior,
    x="horas_extras_mes",
    y="score_satisfacao",
    hue="status_atual",
    palette=["#4C72B0", "#EF4444"],
    errorbar=None,
    ax=ax,
)

max_score = df_junior["score_satisfacao"].max()
plt.ylim(0, max_score * 1.15)

plt.title(
    "Juniores: Impacto das Horas Extras Mensais no Score de Satisfação",
    fontsize=12,
    fontweight="bold",
)

plt.xlabel("Quantidade de Horas Extras Mensais", fontsize=10)
plt.ylabel("Score de Satisfação", fontsize=10)
plt.legend(title="Status", frameon=True)

plt.tight_layout()
plt.savefig("2.4.grafico_satisfacao_horas_juniores.png", dpi=300, bbox_inches="tight")
plt.show()


### Hipótese 3: Os juniores desligados estão concentrados no departamento trabalhista.

todos_departamentos = df_completo["nome_departamento"].unique()

df_junior_desligados = df_completo[
    (df_completo["nivel"] == "Júnior")
    & (df_completo["status_atual"] == "Desligado")
].copy()

fig, ax = plt.subplots(figsize=(8, 5))

sns.countplot(
    data=df_junior_desligados,
    x="nome_departamento",
    order=todos_departamentos,
    hue="nome_departamento",
    palette="Blues_d",
    legend=False,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fontweight="bold", fontsize=10, padding=3)

max_val = (
    df_junior_desligados["nome_departamento"].value_counts().max()
    if not df_junior_desligados.empty
    else 10
)
plt.ylim(0, max_val * 1.18)

plt.title(
    "Distribuição de Juniores Desligados por Departamento",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Departamento", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)

plt.tight_layout()
plt.savefig("2.4.grafico_distribuicao_juniores_desligados.png", dpi=300, bbox_inches="tight")
plt.show()


### Hipótese 4: Plenos saem por não serem promovidos.

df_pleno = df_completo[df_completo["nivel"] == "Pleno"].copy()

if df_pleno["foi_promovido"].dtype in ["int64", "float64", "bool"]:
    df_pleno["status_promocao"] = df_pleno["foi_promovido"].map(
        {1: "Promovido", 0: "Não Promovido", True: "Promovido", False: "Não Promovido"}
    )
else:
    df_pleno["status_promocao"] = df_pleno["foi_promovido"]

fig, ax = plt.subplots(figsize=(8, 5))

sns.countplot(
    data=df_pleno,
    x="status_promocao",
    hue="status_atual",
    hue_order=["Ativo", "Desligado"],
    palette=["#4C72B0", "#EF4444"],
    order=["Não Promovido", "Promovido"],
    ax=ax,
    alpha=1,
)

for container in ax.containers:
    ax.bar_label(container, fontweight="bold", fontsize=10, padding=3)

max_count = (
    df_pleno.groupby(["status_promocao", "status_atual"]).size().max()
)
plt.ylim(0, max_count * 1.18)

plt.title(
    "Nível Pleno: Relação entre Promoção e Evasão de Colaboradores",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Status de Promoção", fontsize=10)
plt.ylabel("Quantidade de Funcionários", fontsize=10)
plt.legend(title="Status", frameon=True)

plt.tight_layout()
plt.savefig("2.4.grafico_plenos_promocao_vs_evasao.png", dpi=300, bbox_inches="tight")
plt.show()

### Hipótese 6: Sêniores estão saindo por não trabalharem no Home Office.
df_senior = df_completo[df_completo["nivel"] == "Sênior"].copy()

df_senior["ano_admissao"] = pd.to_datetime(df_senior["data_admissao"]).dt.year

fig, ax = plt.subplots(figsize=(10, 5))

sns.countplot(
    data=df_senior,
    x="ano_admissao",
    hue="status_atual",
    hue_order=["Ativo", "Desligado"],
    palette={"Ativo": "#4C72B0", "Desligado": "#EF4444"},
    alpha=1,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fontweight="bold", fontsize=9, padding=3)

max_count = (
    df_senior.groupby(["ano_admissao", "status_atual"]).size().max()
    if not df_senior.empty
    else 10
)
plt.ylim(0, max_count * 1.18)

plt.title(
    "Nível Sênior: Volume de Ativos vs. Desligados por Ano de Admissão",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("Ano de Admissão (Safra)", fontsize=10)
plt.ylabel("Quantidade de Colaboradores", fontsize=10)
plt.legend(title="Status", frameon=True)

plt.tight_layout()
plt.savefig(
    "2.4.grafico_seniores_evasao_por_ano_admissao.png", dpi=300, bbox_inches="tight"
)
plt.show()



